# Threshold autoregression: the lynx and the sunspot cycles

Why does the Canadian lynx population climb for six years and crash in three, and why does sunspot activity rise faster than it falls? A linear autoregression cannot say: its cycles are symmetric, and without noise they die out. Tong and Lim (1980) let the autoregression change with the level of the series itself, one linear model below a threshold and another above it, and so produce asymmetric cycles that do not die out. Their three models, eqs. 9.1 to 9.3, with their limit cycles, variances, NAICs and prediction errors, are printed beside Julia's.

The data is `lynxmackenzie`, from PollisDatasets: the number of lynx trapped in the Mackenzie River district of Canada each year from 1821 to 1934. The sunspot numbers are Wolf's yearly numbers as distributed with R (`sunspot.year`), the values the paper used: SILSO has since revised them, and its licence (CC BY-NC 4.0) does not allow them to be distributed with Pollis, so they are read from R's source into memory and not saved. The threshold autoregression comes from the ThresholdAR package, of Compose › New Methods: no registered Julia package fits it.

In [ ]:
Pollis.packages("ThresholdAR" => "https://github.com/Trumpingtons/ThresholdAR.jl", "PollisDatasets", "Plots")

## Set-up

The lynx counts as logarithms to the base 10, `x`, as in the paper; the sunspot numbers of 1700 to 1955, `spots`, with `year` giving the index of a year; `using ThresholdAR` for the threshold autoregression, with `bestregime`, `design`, `shortfit` and `rmse` from it; and `replicate`, which prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using ThresholdAR, PollisDatasets, Plots, Printf, Statistics, Downloads
using ThresholdAR: bestregime, design, shortfit, rmse

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

lynx = dataset("lynxmackenzie");  # one row per year, 1821-1934
x = log10.(Float64.(lynx.lynx));
url = "https://raw.githubusercontent.com/wch/r-source/7bcd838b66763d5029b5bdec668e3436c6e38017/src/library/datasets/data/sunspot.year.R"
text = String(take!(Downloads.download(url, IOBuffer())));  # in memory only, never saved
spots = parse.(Float64, [m.match for m in eachmatch(r"\d+\.\d+", text)])[1:256];  # 1700-1955
year(y) = y - 1699;  # the index of a year in spots

## 1. The data

Tong and Lim (p. 259) see two features in the lynx: (i) cycles of about 10 years with varying amplitudes, and (ii) a rise, from a local minimum to the next maximum, longer than the descent to the following minimum, "thereby showing time irreversibility". Both can be counted in the data: `peaks` finds the peaks, and the rise to each is from the lowest point since the one before. Sunspot activity, too, rises faster than it falls (p. 261).

In [ ]:
"""
	peaks(x; w = 3)

The peaks of `x`: the points highest within `w` steps on either side, so that small wiggles do not count.
"""
peaks(x; w=3) = [t for t in w+1:length(x)-w if x[t] == maximum(x[t-w:t+w])]

for (name, series) in [("lynx, 1821-1934", x), ("sunspots, 1700-1955", spots)]
	p = peaks(series)
	troughs = [p[i] - 1 + argmin(series[p[i]:p[i+1]]) for i in 1:length(p)-1]  # the lowest point between two peaks
	@printf("%-22s %2d peaks, %.1f years apart on average: rise %.1f years, descent %.1f\n",
		name, length(p), (p[end] - p[1]) / (length(p) - 1), mean(p[2:end] - troughs), mean(troughs - p[1:end-1]))
end

plot(plot(1821:1934, x, label="", ylabel="log10 lynx trapped", marker=:circle, ms=2),
	plot(1700:1955, spots, label="", ylabel="sunspot number", marker=:circle, ms=1.5), layout=(2, 1), xlabel="year", size=(700, 450))

## 2. A linear autoregression

The linear model is the starting point: an autoregression of order k, fitted by least squares, with the order chosen by AIC, n ln(RSS/n) + 2(k + 1) (Tong and Lim's eq. 8.3). The package's `bestregime` fits the orders up to the largest to the observations at `times`, so that models of different orders are fitted to the same observations, and returns the one with the smallest AIC; `design` gives the regressors, from which the predictions follow. For the lynx, with orders up to 10, AIC chooses an AR(10).

The eventual forecasting function is the series continued by the model without noise (Section 7). For a stationary linear autoregression it dies out to the model's mean: the cycle the data shows is not in the model, only in its noise.

In [ ]:
linear = bestregime(x, 11:114, 10)  # orders up to 10, so fitted from 1831
@printf("lynx: AR(%d), NAIC %.5f\n", linear.k, linear.aic / linear.n)

y = copy(x)  # continued without noise
for _ in 1:300
	push!(y, only(design(y, [length(y) + 1], linear.k) * linear.b))
end
span(v) = maximum(v) - minimum(v)
@printf("eventual forecasting function: range %.4f over years 1-20 ahead, %.6f over years 281-300; value at 300 %.4f, the model's mean %.4f\n",
	span(y[115:134]), span(y[end-19:end]), y[end], linear.b[1] / (1 - sum(linear.b[2:end])))
plot(1821:2034, y[1:214], label="AR(10): data, then without noise", xlabel="year", ylabel="log10 lynx", size=(700, 300))
hline!([linear.b[1] / (1 - sum(linear.b[2:end]))], label="the model's mean", linestyle=:dash)

## 3. The threshold autoregression

In SETAR(2; k1, k2), with delay d and threshold r, the series follows an autoregression of order k1 when x(t-d) <= r and of order k2 when x(t-d) > r (Tong and Lim, Section 8, p. 257). Each regime is fitted by least squares to its own observations, from observation max(d, L) + 1, L the largest order considered, so that every model compared is fitted to the same observations. Its AIC is the sum of the two regimes' (eq. 8.4), and the normalised AIC, NAIC, is the AIC divided by the number of observations fitted (eq. 8.6), which compares fits to different numbers of observations.

The package's `setar` fits it, `predict` gives one-step-ahead predictions, `eventual` continues the series by the model without noise, and `limitcycle` finds the periodic function it settles into, from its smallest value (Definition 7.1); `risedescent` gives the rise and descent of each of its local cycles.

## 4. Identification

Tong and Lim's identification (pp. 257-258): for a delay d, the candidate thresholds are the 30, 40, 50, 60 and 70 per cent points of x(t-d); for each, the order of each regime is the one with the smallest AIC, and the threshold chosen is the one with the smallest total. Delays are compared by NAIC, and the final choice also weighs the eventual forecasting function and the prediction errors of held-out observations (pp. 258-259).

`identify` gives the model chosen for a delay, and with a single quantile the candidate at that threshold; `selectdelay` runs it for each delay. For the lynx, with orders up to 10, the search does not by itself arrive at the published model: the smallest NAIC is at d = 3. At d = 2 the published threshold, 3.1163, is one of the candidates, the 60 per cent point of x(t-2): log10 of 1,307, the count of 1902.

In [ ]:
describe(m) = @sprintf("d = %d, r = %.4f, SETAR(2; %d, %d), NAIC %.5f", m.d, m.r, m.orders..., m.naic)
println("Lynx, the model identified for each delay:")
foreach(m -> println("  ", describe(m)), selectdelay(x; maxorder=10, delays=1:6).models)
println("\nAt d = 2, each candidate threshold:")
foreach(q -> println("  ", describe(identify(x, 2; maxorder=10, quantiles=[q]))), 0.3:0.1:0.7)
@printf("\n10**3.1163 = %.1f; the lynx count of 1902: %d\n", 10^3.1163, lynx.lynx[findfirst(==(1902), lynx.year)])

## 5. The lynx: eq. 9.1 and Fig. 8

Tong and Lim's model for the lynx is SETAR(2; 8, 3) with d = 2 and r = 3.1163, fitted with orders up to 10, so from 1831 (p. 259). Its eventual forecasting function settles into a limit cycle of period 9 that rises for six years and falls for three: the asymmetry of the data, which the linear model of section 2 lost. The paper lists the cycle from its smallest value backwards (section 10).

In [ ]:
"""Replicate the coefficients of a SETAR model, given as printed, regime by regime."""
function coefficients(title, m, published)
	replicate(title, [("regime $j, " * (i == 1 ? "constant" : "x(t-$(i - 1))"), b, published[j][i]) for j in 1:2 for (i, b) in enumerate(m.coefficients[j])])
end

m1 = setar(x, 2, 3.1163, (8, 3); maxorder=10)
coefficients("Eq. 9.1, lynx (Tong and Lim 1980, p. 259)", m1, (
	["0.5239", "1.0359", "-0.1756", "0.1753", "-0.4339", "0.3457", "-0.3032", "0.2165", "0.0043"],
	["2.6559", "1.4246", "-1.1618", "-0.1094"]))
replicate("Fit (Fig. 8, p. 260)", [("root mean RSS", sqrt(sum(m1.rss) / length(m1.fitted)), "0.18939")])

c1 = limitcycle(m1, x)
replicate("Limit cycle, in the order printed (p. 259)", collect(zip(string.(1:9), [c1[1]; reverse(c1[2:end])],
	["2.6226", "2.8945", "3.2523", "3.4601", "3.4257", "3.2281", "2.9793", "2.7884", "2.6639"])))
println("\nperiod ", length(c1), "; rise and descent ", risedescent(c1), " (p. 259: period 9, rise six years, descent three)")

plot(1821:1934, x, label="real data", marker=:+, ms=3, xlabel="year", ylabel="log10 lynx", size=(700, 330), ylims=(0.8, 4), legend=:bottomleft)
plot!(1831:1934, predict(m1, x, m1.fitted), label="from fitted model", marker=:circle, ms=1.5)
plot!(1935:2000, eventual(m1, x, 66), label="eventual forecasting function", marker=:square, ms=1.5)
hline!([3.1163], label="threshold 3.1163", linestyle=:dash)

## 6. Gain spectra: Figs. 9 and 11

Each regime is a linear autoregression, and its gain spectrum, -10 log10 |1 - a1 e^(-2 pi i f) - ... - ak e^(-2 pi i f k)|^2 in decibels, shows the frequencies it amplifies (p. 260). Drawn relative to its peak, as in Fig. 9. For the lynx the two regimes peak at different frequencies, which Tong and Lim read as an "amplitude-frequency dependency"; the upper regime peaks near 0.13, a cycle of about 8 years. The text says Figs. 9a and 9b are the upper and lower regimes, in that order; by their shapes they are the other way round (section 10).

In [ ]:
gain(b, f) = -10log10(abs2(1 - sum(b[k + 1] * cis(-2pi * f * k) for k in 1:length(b) - 1)))
frequencies = 0:0.0025:0.5
m2 = setar(spots[year(1700):year(1920)], 3, 36.6, (4, 12); maxorder=20)  # eq. 9.2, section 7
panels = [begin
		g = [gain(m.coefficients[j], f) for f in frequencies]
		plot(frequencies, g .- maximum(g), label="", title=title, titlefontsize=9, ylims=(-50, 2), xlabel="frequency", ylabel="dB")
	end for (m, j, title) in [(m1, 1, "Fig. 9a: lynx, x(t-2) <= 3.1163"), (m1, 2, "Fig. 9b: lynx, x(t-2) > 3.1163"),
		(m2, 1, "Fig. 11a: sunspots, x(t-3) <= 36.6"), (m2, 2, "Fig. 11b: sunspots, x(t-3) > 36.6")]]
for (name, m) in [("lynx", m1), ("sunspots", m2)], j in 1:2
	g = [gain(m.coefficients[j], f) for f in frequencies]
	@printf("%-9s regime %d: peak at frequency %.3f\n", name, j, frequencies[argmax(g)])
end
plot(panels..., layout=(2, 2), size=(700, 450))

## 7. Sunspots, 1700-1920: eq. 9.2 and Fig. 10

SETAR(2; 4, 12) with d = 3 and r = 36.6, fitted with orders up to 20, so from 1720 (p. 261). It predicts 1921 to 1955 one step ahead, and its eventual forecasting function is periodic with a period of 31 years made of three local cycles, each rising for 4 years and falling for 6, 6 and 7: "a fractional period" of 31/3 years. Their asymmetry runs in the reverse direction to the lynx's.

In [ ]:
coefficients("Eq. 9.2, sunspots 1700-1920 (p. 261)", m2, (
	["10.5440", "1.6920", "-1.1592", "0.2367", "0.1503"],
	["7.8041", "0.7432", "-0.0409", "-0.2020", "0.1730", "-0.2266", "0.0189", "0.1612", "-0.2564", "0.3195", "-0.3891", "0.4306", "-0.0397"]))
p2 = predict(m2, spots, year(1921):year(1955))
replicate("One-step-ahead predictions (Fig. 10, p. 261)", [("RMSE 1921-1955", rmse(spots[year(1921):year(1955)] - p2), "12.17397")])
c2 = limitcycle(m2, spots[year(1700):year(1920)])
replicate("Eventual forecasting function (p. 261)", [("period", length(c2), "31"),
	[(name, v, p) for (name, v, p) in zip(["rise 1", "descent 1", "rise 2", "descent 2", "rise 3", "descent 3"], collect(Iterators.flatten(risedescent(c2))), ["4", "6", "4", "6", "4", "7"])]...])

plot(1700:1955, spots, label="real data", marker=:+, ms=2, xlabel="year", ylabel="sunspot number", size=(700, 330), legend=:topleft)
plot!(1720:1920, predict(m2, spots, m2.fitted), label="from fitted model")
plot!(1921:1955, p2, label="prediction (one step ahead)", marker=:circle, ms=2)
plot!(1956:2020, eventual(m2, spots, 65), label="eventual forecasting function", marker=:square, ms=1.5)
hline!([36.6], label="threshold 36.6", linestyle=:dash)

## 8. Sunspots, 1837-1924: eq. 9.3 and Fig. 12

Part of the sunspot record's non-stationarity must come from the photographic recording introduced towards its later part (p. 262), so Tong and Lim fit the years 1837 to 1924 alone: SETAR(2; 4, 2) with d = 5 and r = 47.4, with orders up to 10, so from 1847. The one-step-ahead predictions are reasonable up to 1944 "but deteriorate rapidly from then on", which Julia's errors show.

In [ ]:
m3 = setar(spots[year(1837):year(1924)], 5, 47.4, (4, 2); maxorder=10)
coefficients("Eq. 9.3, sunspots 1837-1924 (p. 262)", m3, (["25.2120", "0.9820", "-0.0377", "-0.6378", "0.2454"], ["0.3585", "0.7569", "-0.0531"]))
p3 = predict(m3, spots, year(1925):year(1955))
replicate("One-step-ahead predictions (Fig. 12, p. 262)", [("RMSE 1925-1944", rmse(spots[year(1925):year(1944)] - p3[1:20]), "9.2342")])
@printf("\nRMSE 1945-1955: %.4f, after the period the paper reports\n", rmse(spots[year(1945):year(1955)] - p3[21:end]))

plot(1837:1955, spots[year(1837):end], label="real data", marker=:+, ms=2, xlabel="year", ylabel="sunspot number", size=(700, 330), legend=:topleft)
plot!(1925:1955, p3, label="prediction (one step ahead)", marker=:circle, ms=2)
plot!(1956:2025, eventual(m3, spots, 70), label="eventual forecasting function", marker=:square, ms=1.5)
hline!([47.4], label="threshold 47.4", linestyle=:dash)

## 9. The branch: linear or threshold?

The threshold model is worth its extra parameters if it fits better for its size, a smaller NAIC than the best linear autoregression fitted to the same observations, and predicts observations it was not fitted to better. The held-out years are those of Figs. 10 and 12 for the sunspots and, for the lynx, the last 10 per cent of the series, 1924 to 1934, as in Tong and Lim's pseudo-cross-validation (p. 259), with both models refitted without them. If the threshold model wins on both counts for every series, `threshold` is true and the SETAR models are the ones to keep; otherwise the linear ones.

In [ ]:
predictar(f, x, times) = design(x, times, f.k) * f.b  # one step ahead, by the linear fit f
kept = findall(<=(1923), lynx.year)  # 1821-1923
held = findall(>=(1924), lynx.year)  # 1924-1934, the last 10 per cent
lynxfit = setar(x[kept], 2, 3.1163, (8, 3); maxorder=10)
lynxar = bestregime(x[kept], lynxfit.fitted, 10)
rows = [
	("Lynx (eq. 9.1)", m1.naic, bestregime(x, m1.fitted, 10), rmse(x[held] - predict(lynxfit, x, held)), rmse(x[held] - predictar(lynxar, x, held)), "1924-1934"),
	("Sunspots (eq. 9.2)", m2.naic, bestregime(spots[1:year(1920)], m2.fitted, 20), rmse(spots[year(1921):year(1955)] - p2),
		rmse(spots[year(1921):year(1955)] - predictar(bestregime(spots[1:year(1920)], m2.fitted, 20), spots, year(1921):year(1955))), "1921-1955"),
	("Sunspots (eq. 9.3)", m3.naic, bestregime(spots[year(1837):year(1924)], m3.fitted, 10), rmse(spots[year(1925):year(1944)] - p3[1:20]),
		rmse(spots[year(1925):year(1944)] - predictar(bestregime(spots[year(1837):year(1924)], m3.fitted, 10), spots, year(1925):year(1944))), "1925-1944")]
@printf("%-20s%12s%12s%8s%14s%14s  %s\n", "", "SETAR NAIC", "AR NAIC", "AR(k)", "SETAR RMSE", "AR RMSE", "held out")
for (name, naic, linear, e1, e2, years) in rows
	@printf("%-20s%12.5f%12.5f%8d%14.5f%14.5f  %s\n", name, naic, linear.aic / linear.n, linear.k, e1, e2, years)
end
threshold = all(r -> r[2] < r[3].aic / r[3].n && r[4] < r[5], rows)

## 10. What differs, and why

All 39 coefficients of eqs. 9.1 to 9.3, the lynx root mean RSS, the sunspot prediction error of Fig. 10, the sunspot period and its local cycles agree with the published values to every printed digit. The rest:

- **The variances and NAICs are those of a smaller model.** The residual variances printed under each equation, and the NAICs of Figs. 8, 10 and 12, are those of the same models with one lag fewer in each regime, with the NAIC penalised as if they had the orders stated (the cell below). All three models show it: computed that way, 11 of the 12 values agree to every printed digit, and eq. 9.3's NAIC, 5.08178, is printed 5.0817, truncated rather than rounded. For the models as stated the NAICs are -3.13819, 4.99430 and 5.06802. The coefficients and the prediction errors are those of the models as stated: the 1921-1955 RMSE, 12.17397, is SETAR(2; 4, 12)'s; SETAR(2; 3, 11) gives 12.35812. It looks like an off-by-one in the program that printed the summaries.
- **The lynx cycle is printed backwards.** Read forwards, the printed cycle rises for three years and falls for six; the text, and the model, rise for six and fall for three. The paper lists the cycle from its smallest value backwards.
- **Figs. 9a and 9b are swapped in the text.** The text gives them as the regimes x(t-2) > 3.1163 and x(t-2) <= 3.1163, in that order. By their shapes, 9a, highest at frequency 0 with a second bump near 0.37, is the lower regime, the AR(8), and 9b, peaking near 0.13, the upper one, the AR(3) (section 6).
- **One unit in the last digit.** The lynx limit cycle's sixth value, 3.22805, printed 3.2281, and eq. 9.3's RMSE, 9.23414, printed 9.2342.
- **Not checked: the 10 per cent of Tong (1980a).** The paper quotes Tong (1980a): SETAR(2; 6, 3) with d = 2, fitted to 1821-1920, reduces the root mean square error of one-step-ahead predictions by 10 per cent against the AR(12) of Tong (1977b). The paper does not give that model's threshold; with the candidate thresholds of section 4, the reduction against an AR(12) fitted by least squares to the same years is 13 to 39 per cent, not 10.
- **The identification.** As section 4 shows, the search alone gives d = 3 for the lynx; the published d = 2 was chosen with the eventual forecasting function, the prediction errors and judgement as well.

In [ ]:
labels = ["var, regime 1", "var, regime 2", "pooled", "NAIC"]
replicate("Eq. 9.1 and Fig. 8, lynx (pp. 259-260)", collect(zip(labels, shortfit(x, 2, 3.1163, (8, 3), 10), ["0.0255", "0.0516", "0.0360", "-3.13559"])))
replicate("Eq. 9.2 and Fig. 10, sunspots 1700-1920 (p. 261)", collect(zip(labels, shortfit(spots[year(1700):year(1920)], 3, 36.6, (4, 12), 20), ["254.64", "66.80", "153.71", "4.99995"])))
replicate("Eq. 9.3 and Fig. 12, sunspots 1837-1924 (p. 262)", collect(zip(labels, shortfit(spots[year(1837):year(1924)], 5, 47.4, (4, 2), 10), ["231.030", "63.075", "157.819", "5.0817"])))
@printf("\nNAIC of the models as stated: %.5f, %.5f, %.5f\n", m1.naic, m2.naic, m3.naic)
@printf("RMSE 1921-1955 of SETAR(2; 3, 11): %.5f\n", rmse(spots[year(1921):year(1955)] - predict(setar(spots[year(1700):year(1920)], 3, 36.6, (3, 11); maxorder=20), spots, year(1921):year(1955))))

**Now in Compose › New Methods.** No Julia package fits a threshold autoregression. The New Methods package **ThresholdAR**, used here, has it as documented functions (fit, identification, delay selection, prediction, eventual forecasting function and limit cycles), tested against the paper's three models. Find it in **Compose › New Methods**. The paper's other examples, the TARSO model of the mink-muskrat data and the TARSC models, use data the paper does not give in full.

## References

- Campbell, M. J. and Walker, A. M. (1977). A survey of statistical work on the Mackenzie River series of annual Canadian lynx trappings for the years 1821-1934 and a new analysis. *Journal of the Royal Statistical Society: Series A* 140(4), 411-431.
- Elton, C. and Nicholson, M. (1942). The ten-year cycle in numbers of the lynx in Canada. *Journal of Animal Ecology* 11(2), 215-244.
- Tong, H. (1977). Some comments on the Canadian lynx data. *Journal of the Royal Statistical Society: Series A* 140(4), 432-436.
- Tong, H. and Lim, K. S. (1980). Threshold autoregression, limit cycles and cyclical data. *Journal of the Royal Statistical Society: Series B* 42(3), 245-268. doi:10.1111/j.2517-6161.1980.tb01126.x
- Data: the lynx counts from PollisDatasets (`lynxmackenzie`); the sunspot numbers as distributed with R (`sunspot.year`), read from R's source and not saved, since SILSO's licence does not allow them to be distributed with Pollis.